In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/raw")
REPORT_DIR = Path("../reports")

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Data folder exists:", DATA_DIR.exists())
print("Reports folder exists:", REPORT_DIR.exists())

for file in sorted(DATA_DIR.glob("*.csv")):
    print(file.name)

Data folder exists: True
Reports folder exists: True
campaign_data.csv
coupon_item_mapping.csv
customer_demographics.csv
customer_transaction_data.csv
item_data.csv
sample_submission_Byiv0dS.csv
test_QyjYwdj.csv
train.csv


In [2]:
data = {
    file.stem: pd.read_csv(file)
    for file in DATA_DIR.glob("*.csv")
}

train = data["train"]
campaigns = data["campaign_data"]
customers = data["customer_demographics"]
transactions = data["customer_transaction_data"]
coupon_items = data["coupon_item_mapping"]
items = data["item_data"]

checks = pd.DataFrame([
    {
        "relationship": "Training → Campaign",
        "matched_rows": train["campaign_id"].isin(
            campaigns["campaign_id"]
        ).sum(),
        "total_rows": len(train)
    },
    {
        "relationship": "Training → Customer demographics",
        "matched_rows": train["customer_id"].isin(
            customers["customer_id"]
        ).sum(),
        "total_rows": len(train)
    },
    {
        "relationship": "Training → Coupon mapping",
        "matched_rows": train["coupon_id"].isin(
            coupon_items["coupon_id"]
        ).sum(),
        "total_rows": len(train)
    },
    {
        "relationship": "Coupon mapping → Item data",
        "matched_rows": coupon_items["item_id"].isin(
            items["item_id"]
        ).sum(),
        "total_rows": len(coupon_items)
    }
])

checks["unmatched_rows"] = (
    checks["total_rows"] - checks["matched_rows"]
)

checks["match_percentage"] = (
    checks["matched_rows"] / checks["total_rows"] * 100
).round(2)

display(checks)

,relationship,matched_rows,total_rows,unmatched_rows,match_percentage
0,Training → Campaign,78369,78369,0,100.00
1,Training → Customer demographics,43661,78369,34708,55.71
2,Training → Coupon mapping,78369,78369,0,100.00
3,Coupon mapping → Item data,92663,92663,0,100.00


In [3]:
checks.to_csv(
    REPORT_DIR / "day03_relationship_validation.csv",
    index=False
)

print("Day 3 relationship report saved successfully.")

Day 3 relationship report saved successfully.


In [4]:
customer_match = train["customer_id"].isin(
    customers["customer_id"]
)

unmatched_customers = train.loc[
    ~customer_match,
    ["customer_id"]
].drop_duplicates()

print("Total training records:", len(train))
print("Records without demographic match:", (~customer_match).sum())
print("Unique unmatched customer IDs:", len(unmatched_customers))
print("Unique customers in training:", train["customer_id"].nunique())
print("Unique customers in demographics:", customers["customer_id"].nunique())

display(unmatched_customers.head(10))

Total training records: 78369
Records without demographic match: 34708
Unique unmatched customer IDs: 725
Unique customers in training: 1428
Unique customers in demographics: 760


,customer_id
3,1050
8,523
14,761
21,1255
25,63
26,88
28,1095
29,1096
30,919
34,91


In [5]:
print("Training rows:", len(train))
print("Unique training IDs:", train["id"].nunique())
print("Unique campaign IDs in training:", train["campaign_id"].nunique())
print("Unique coupon IDs in training:", train["coupon_id"].nunique())
print("Unique customer IDs in training:", train["customer_id"].nunique())

print("\nDuplicate training IDs:", train["id"].duplicated().sum())

Training rows: 78369
Unique training IDs: 78369
Unique campaign IDs in training: 18
Unique coupon IDs in training: 866
Unique customer IDs in training: 1428

Duplicate training IDs: 0


In [6]:
import pandas as pd

training_summary = pd.DataFrame({
    "Metric": [
        "Total Training Records",
        "Unique Training IDs",
        "Unique Campaign IDs",
        "Unique Coupon IDs",
        "Unique Customer IDs",
        "Duplicate Training IDs"
    ],
    "Value": [
        len(train),
        train["id"].nunique(),
        train["campaign_id"].nunique(),
        train["coupon_id"].nunique(),
        train["customer_id"].nunique(),
        train["id"].duplicated().sum()
    ]
})

display(training_summary)

training_summary.to_csv(
    REPORT_DIR / "day03_training_structure.csv",
    index=False
)

print("Day 3 training structure report saved successfully!")

,Metric,Value
0,Total Training Records,78369
1,Unique Training IDs,78369
2,Unique Campaign IDs,18
3,Unique Coupon IDs,866
4,Unique Customer IDs,1428
5,Duplicate Training IDs,0


Day 3 training structure report saved successfully!
